# Data Modalities in Diffusion Models
**Lecture — Politecnico di Milano**

Diffusion models are remarkably general. The **same algorithm** that generates images also generates videos, audio, and robot trajectories.
The key insight: the diffusion process operates on **tensors** — it is agnostic to the semantic meaning of the data.

We explore three main modalities:
1. **Images** — shape `(C, H, W)`, batched `(B, C, H, W)`
2. **Videos** — shape `(T, C, H, W)`, batched `(B, T, C, H, W)`
3. **Robot Trajectories** — shape `(T_h, D)`, batched `(B, T_h, D)`

The diffusion model treats each sample as a vector $\mathbf{x}_0 \in \mathbb{R}^d$ and learns the distribution $p(\mathbf{x}_0)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from PIL import Image
import warnings
import os
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

np.random.seed(42)
torch.manual_seed(42)
print('Libraries loaded successfully.')

---
## 1. Images

An image is a 2-D array of pixels. In PyTorch, images follow the **channel-first** convention:

| Format | Shape | Notes |
|---|---|---|
| NumPy (HWC) | `(H, W, C)` | Standard for Pillow, OpenCV |
| PyTorch (CHW) | `(C, H, W)` | Standard for neural nets |
| Batched | `(B, C, H, W)` | What the model actually sees |

**Pixel values** are typically normalized:
- $[0, 1]$ for display
- $[-1, 1]$ for diffusion models (zero-mean makes the noise schedule cleaner)

In [ ]:
# ── Create a synthetic 64×64 RGB image ─────────────────────────────────────
def make_synthetic_image(size=64):
    img = np.zeros((size, size, 3), dtype=np.float32)
    # Gradient background
    for i in range(size):
        img[i, :, 0] = i / size          # red varies top-to-bottom
        img[:, i, 1] = i / size          # green varies left-to-right
    img[:, :, 2] = 0.4                   # constant blue
    # Yellow circle in the centre
    cx = cy = size // 2
    r = size // 5
    Y, X = np.ogrid[:size, :size]
    mask = (X - cx)**2 + (Y - cy)**2 < r**2
    img[mask] = [1.0, 0.9, 0.1]
    # Small white square in corner
    img[4:14, 4:14] = [1.0, 1.0, 1.0]
    return np.clip(img, 0, 1)

def load_image(path, size):
    if path and os.path.isfile(path):
        img = Image.open(path).convert('RGB').resize((size, size))
        # arr = np.array(img, dtype=np.float32)
        arr = np.array(img, dtype=np.float32) / 255.0
    else:
        if path:
            print(f'[Warning] {path} not found — using synthetic image.')
        arr = make_synthetic_image(size)
    # tensor = torch.from_numpy(arr).permute(2, 0, 1)  # (1,C,H,W)
    # return tensor * 2.0 - 1.0   # [0,1] → [-1,1]
    return arr

IMAGE_PATH = None    # e.g. 'my_photo.jpg'  |  None → synthetic
if IMAGE_PATH is not None:
    image_hwc = load_image(IMAGE_PATH, 64)
else:
    image_hwc = make_synthetic_image(64)
print(f'NumPy image (H, W, C): {image_hwc.shape}')
print(f'Value range: [{image_hwc.min():.2f}, {image_hwc.max():.2f}]')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image_hwc)
axes[0].set_title('RGB Image', fontsize=13, fontweight='bold')
axes[0].axis('off')

channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
cmaps = ['Reds', 'Greens', 'Blues']
for i in range(3):
    im = axes[i+1].imshow(image_hwc[:, :, i], cmap=cmaps[i], vmin=0, vmax=1)
    axes[i+1].set_title(channel_names[i], fontsize=13)
    axes[i+1].axis('off')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046, pad=0.04)

plt.suptitle('Image decomposition into RGB channels — shape (H=64, W=64, C=3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PyTorch tensor format and batching ──────────────────────────────────────
# Convert (H, W, C) → (C, H, W)
tensor_chw = torch.from_numpy(image_hwc).permute(2, 0, 1)
print(f'PyTorch single image (C, H, W): {tuple(tensor_chw.shape)}')
print(f'Total elements  d = C×H×W = {tensor_chw.numel():,}')

# Simulate a batch of 4 images with small brightness variations
brightness_offsets = torch.tensor([-0.15, -0.05, 0.05, 0.15]).view(4, 1, 1, 1)
batch = tensor_chw.unsqueeze(0).repeat(4, 1, 1, 1) + brightness_offsets
batch = batch.clamp(0, 1)
print(f'Batch tensor    (B, C, H, W): {tuple(batch.shape)}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    axes[i].imshow(batch[i].permute(1, 2, 0).numpy())
    axes[i].set_title(f'Sample {i+1}\nshape {tuple(batch[i].shape)}', fontsize=11)
    axes[i].axis('off')
plt.suptitle('Batched images — shape (B=4, C=3, H=64, W=64)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Normalize to [-1, 1] for diffusion
batch_norm = batch * 2 - 1
print(f'\nAfter normalization to [-1,1]: min={batch_norm.min():.2f}, max={batch_norm.max():.2f}')

---
## 2. Videos

A video is a **sequence of frames**. Adding the temporal dimension $T$ gives:

| Format | Shape | Notes |
|---|---|---|
| Single video | `(T, C, H, W)` | T frames |
| Batched | `(B, T, C, H, W)` | B videos of T frames each |

Models like **VideoLDM** and **Sora** operate on this format.  
The total dimensionality is $d = T \times C \times H \times W$, which grows quickly — hence the use of latent spaces.

A useful diagnostic is the **kymograph** (space-time slice): fix one spatial row and plot its pixel values over time to reveal motion.

In [ ]:
# ── Create a synthetic video: orange ball moving left → right ───────────────
def make_synthetic_video(T=8, size=32):
    frames = []
    for t in range(T):
        frame = np.zeros((size, size, 3), dtype=np.float32)
        # Background: blue hue increases over time
        frame[:, :, 2] = 0.1 + 0.6 * (t / (T - 1))
        frame[:, :, 1] = 0.05
        # Moving ball
        cx = int(size * 0.15 + size * 0.70 * t / (T - 1))
        cy = size // 2
        r = max(2, size // 6)
        Y, X = np.ogrid[:size, :size]
        mask = (X - cx)**2 + (Y - cy)**2 < r**2
        frame[mask] = [1.0, 0.55, 0.0]   # orange
        frames.append(frame)
    return np.stack(frames)   # (T, H, W, C)

video = make_synthetic_video(T=8, size=32)
print(f'Video numpy shape (T, H, W, C): {video.shape}')

video_tensor = torch.from_numpy(video).permute(0, 3, 1, 2)  # (T, C, H, W)
print(f'PyTorch video    (T, C, H, W): {tuple(video_tensor.shape)}')

batch_video = video_tensor.unsqueeze(0)                      # (B, T, C, H, W)
print(f'Batched video (B, T, C, H, W): {tuple(batch_video.shape)}')
print(f'Total elements d = T×C×H×W = {video_tensor.numel():,}')

# Show all frames
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for t in range(8):
    axes[t].imshow(video[t])
    axes[t].set_title(f't = {t}', fontsize=12)
    axes[t].axis('off')
plt.suptitle('Video: 8 frames, each (C=3, H=32, W=32) — ball moves left → right', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Robot Trajectories

In **robot learning** (e.g. imitation learning, Diffusion Policy), the data is a sequence of actions:

$$\tau = (\mathbf{a}_0, \mathbf{a}_1, \ldots, \mathbf{a}_{T_h-1}), \quad \mathbf{a}_t \in \mathbb{R}^D$$

| Common action spaces | $D$ |
|---|---|
| 2D end-effector $(x, y)$ | 2 |
| 3D end-effector $(x, y, z)$ | 3 |
| 6-DOF joint angles | 6 |
| 7-DOF arm + gripper | 7 |

The **Diffusion Policy** (Chi et al. 2023) uses a diffusion model conditioned on visual observations to generate action sequences directly.

Shape: `(T_h, D)` for a single trajectory, `(B, T_h, D)` for a batch.

In [ ]:
# ── Generate trajectory datasets ────────────────────────────────────────────
def make_figure_eight(T=100, noise_std=0.02):
    t = np.linspace(0, 2 * np.pi, T)
    x = np.sin(t) + np.random.randn(T) * noise_std
    y = np.sin(t) * np.cos(t) + np.random.randn(T) * noise_std
    return np.stack([x, y], axis=1)   # (T, 2)

def make_joint_trajectory(T=100, n_joints=6):
    t = np.linspace(0, 2 * np.pi, T)
    joints = []
    for j in range(n_joints):
        amp   = np.pi / (j + 2)
        freq  = 0.4 + j * 0.25
        phase = j * np.pi / 3
        joints.append(amp * np.sin(freq * t + phase) + 0.05 * np.random.randn(T))
    return np.stack(joints, axis=1)   # (T, 6)

# Dataset of 20 demonstrated trajectories (like a robot imitation dataset)
n_demos = 20
demos = []
for _ in range(n_demos):
    noise = np.random.uniform(0.02, 0.10)
    offset = np.random.randn(2) * 0.08
    traj = make_figure_eight(T=100, noise_std=noise)
    traj += offset
    demos.append(traj)

traj_2d    = make_figure_eight(T=100, noise_std=0.01)
traj_joint = make_joint_trajectory(T=100, n_joints=6)

batch_traj = torch.tensor(np.stack(demos), dtype=torch.float32)
print(f'Single 2-D trajectory      (T, D): {traj_2d.shape}')
print(f'Single joint trajectory    (T, D): {traj_joint.shape}')
print(f'Batch of trajectories (B, T, D): {tuple(batch_traj.shape)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: 2-D trajectory dataset ─────────────────────────────────────────
for traj in demos:
    axes[0].plot(traj[:, 0], traj[:, 1], alpha=0.25, color='steelblue', linewidth=1)
axes[0].plot(traj_2d[:, 0], traj_2d[:, 1], 'r-', linewidth=2.5, label='Example traj', zorder=5)
axes[0].scatter(traj_2d[0, 0], traj_2d[0, 1], c='limegreen', s=120, zorder=6, label='Start')
axes[0].scatter(traj_2d[-1, 0], traj_2d[-1, 1], c='red', s=120, marker='*', zorder=6, label='End')
axes[0].set_title('2-D End-Effector Trajectory Dataset\n(T=100, D=2)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('x position')
axes[0].set_ylabel('y position')
axes[0].legend(fontsize=9)
axes[0].set_aspect('equal')

# ── Panel 2: Joint angles over time ─────────────────────────────────────────
t_axis = np.arange(100)
colors = plt.cm.tab10(np.linspace(0, 0.6, 6))
for j in range(6):
    axes[1].plot(t_axis, traj_joint[:, j], color=colors[j], linewidth=1.5, label=f'Joint {j+1}')
axes[1].set_title('6-DOF Joint Angle Trajectories\n(T=100, D=6)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Angle (rad)')
axes[1].legend(fontsize=9, ncol=2)

# ── Panel 3: Dimensionality comparison ──────────────────────────────────────
labels = ['MNIST\nimage\n(1×28×28)', 'RGB image\n(3×64×64)', 'Video clip\n(3×8×32×32)', 'Joint traj\n(T=100,D=6)', '2-D traj\n(T=100,D=2)']
dims   = [784, 12288, 24576, 600, 200]
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = axes[2].bar(labels, dims, color=colors_bar, edgecolor='black', linewidth=0.6)
for bar, d in zip(bars, dims):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 150,
                 str(d), ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[2].set_title('Total Dimensionality $d$ Comparison', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Total dimensions $d$')
axes[2].tick_params(axis='x', labelsize=8)

plt.suptitle('Robot Trajectories and Data Dimensionality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Normalization

Diffusion models assume $\mathbf{x}_0$ has roughly zero mean and unit variance — matching the prior $\mathcal{N}(\mathbf{0}, \mathbf{I})$ at the end of the forward process.

| Data type | Normalization |
|---|---|
| Image pixels $[0,255]$ | Divide by 127.5, subtract 1 → $[-1, 1]$ |
| Image pixels $[0,1]$ | Multiply by 2, subtract 1 → $[-1, 1]$ |
| Trajectory | Standardize: $(x - \mu) / \sigma$ per dimension |

In [ ]:
# ── Normalization examples ───────────────────────────────────────────────────

# Image: [0,1] → [-1,1]
img_raw   = torch.from_numpy(image_hwc).permute(2, 0, 1)   # (C,H,W) in [0,1]
img_norm  = img_raw * 2.0 - 1.0                             # [-1,1]

# Trajectory: standardize per dimension
traj_raw   = batch_traj.numpy()                             # (B, T, D)
mean_traj  = traj_raw.mean(axis=(0, 1), keepdims=True)      # (1, 1, D)
std_traj   = traj_raw.std(axis=(0, 1), keepdims=True) + 1e-8
traj_norm  = (traj_raw - mean_traj) / std_traj

print('Image normalization:')
print(f'  raw  → min={img_raw.min():.2f}, max={img_raw.max():.2f}, mean={img_raw.mean():.3f}')
print(f'  norm → min={img_norm.min():.2f}, max={img_norm.max():.2f}, mean={img_norm.mean():.3f}')
print('\nTrajectory normalization:')
print(f'  raw  → mean per dim: {traj_raw.mean(axis=(0,1)).round(3)}')
print(f'  norm → mean per dim: {traj_norm.mean(axis=(0,1)).round(3)}')
print(f'  norm → std  per dim: {traj_norm.std(axis=(0,1)).round(3)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Image pixel histogram
axes[0].hist(img_raw.numpy().flatten(), bins=60, alpha=0.6, color='royalblue', label='raw [0,1]', density=True)
axes[0].hist(img_norm.numpy().flatten(), bins=60, alpha=0.6, color='tomato',   label='norm [-1,1]', density=True)
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.2)
axes[0].set_title('Image Pixel Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pixel value')
axes[0].set_ylabel('Density')
axes[0].legend()

# Trajectory distribution
axes[1].hist(traj_raw.flatten(),  bins=60, alpha=0.6, color='royalblue', label='raw',  density=True)
axes[1].hist(traj_norm.flatten(), bins=60, alpha=0.6, color='tomato',    label='norm', density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].set_title('Trajectory Value Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Effect of Normalization (centered near 0 matches the diffusion prior)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Modality | PyTorch shape | $d$ (example) | Key papers |
|---|---|---|---|
| Grayscale image | `(1, H, W)` | 784 | DDPM, Score SDE |
| RGB image | `(3, H, W)` | 12,288 | LDM, DALL·E 2 |
| Video | `(T, C, H, W)` | 24,576 | VideoLDM, Sora |
| Robot trajectory | `(T_h, D)` | 600 | Diffusion Policy |

**Key takeaways**
- The diffusion forward/reverse process is identical across modalities.
- The **neural network architecture** is what changes (U-Net for images, Transformer for sequences).
- Normalization is critical: all modalities must be scaled to approximately $\mathcal{N}(0, I)$.
- Dimensionality matters for compute: video and high-res images require latent compression (LDM).